In [2]:
import pickle
import numpy as np
import pandas as pd
import yfinance as yf
from valuation_utils import build_features, fetch_growth_features


def evaluate_stocks(tickers: list[str], model_file: str = "nifty_valuation_model.pkl"):
    with open(model_file, "rb") as f:
        saved = pickle.load(f)

    rows = []
    for t in tickers:
        try:
            info = yf.Ticker(t).info
            pe = info.get('trailingPE')
            pe = float(pe) if pe is not None else np.nan
            price = info.get('currentPrice', info.get('regularMarketPrice', np.nan))

            row = {
                'Sector': info.get('sector'),
                'ROE': info.get('returnOnEquity', np.nan),
                'ROA': info.get('returnOnAssets', np.nan),
                'Operating_Margin': info.get('operatingMargins', np.nan),
                'Net_Profit_Margin': info.get('profitMargins', np.nan),
                'Revenue_Growth': info.get('revenueGrowth', np.nan),
                'Earnings_Growth': info.get('earningsGrowth', np.nan),
                'Debt_To_Equity': info.get('debtToEquity', np.nan),
                'Current_Ratio': info.get('currentRatio', np.nan),
                'Free_Cash_Flow': info.get('freeCashflow', np.nan),
                'Total_Revenue': info.get('totalRevenue', np.nan)
            }
            row.update(fetch_growth_features(t, info=info))

            X = build_features(pd.DataFrame([row]))[saved['features']]
            fair_pe = float(saved['model'].predict(X)[0])
            lo, hi = fair_pe * np.exp(saved['q_lo']), fair_pe * np.exp(saved['q_hi'])

            if pd.isna(pe) or pe <= 0:
                verdict = "Invalid P/E"
            elif pe < lo:
                verdict = "Below range"
            elif pe > hi:
                verdict = "Above range"
            else:
                verdict = "Within range"

            rows.append({
                'Ticker': t, 'Sector': row['Sector'], 'Price': price,
                'Actual_PE': pe, 'Fair_PE': fair_pe, 'PE_min': lo, 'PE_max': hi,
                'PE/Fair': pe / fair_pe if fair_pe else np.nan,
                'Missing_Features': int(X.isna().sum().sum()),
                'Verdict': verdict
            })
        except Exception as e:
            rows.append({'Ticker': t, 'Verdict': f"Error: {e}"})

    return pd.DataFrame(rows)


sample_tickers = [
    "TCS.NS", "INFY.NS", "HDFCBANK.NS", "SBIN.NS", "RELIANCE.NS",
    "HINDUNILVR.NS", "ITC.NS", "SUNPHARMA.NS", "LT.NS",
    "TATASTEEL.NS", "MARUTI.NS", "NTPC.NS"
]

results = evaluate_stocks(sample_tickers)
pd.set_option('display.width', 200)
print(results.round(2).sort_values('PE/Fair').to_string(index=False))

       Ticker             Sector    Price  Actual_PE  Fair_PE  PE_min  PE_max  PE/Fair  Missing_Features      Verdict
      INFY.NS         Technology  1171.60      15.60    29.52   14.22   64.53     0.53                 0 Within range
      NTPC.NS          Utilities   322.70      10.53    15.61    7.51   34.11     0.67                 0 Within range
      SBIN.NS Financial Services  1016.00      10.37    13.95    6.72   30.49     0.74                 3 Within range
  HDFCBANK.NS Financial Services   784.10      17.14    20.16    9.70   44.05     0.85                 3 Within range
       ITC.NS Consumer Defensive   283.45      18.81    21.30   10.26   46.56     0.88                 0 Within range
    MARUTI.NS  Consumer Cyclical 12815.00      29.95    30.59   14.73   66.86     0.98                 0 Within range
       TCS.NS         Technology  2498.80      17.52    17.71    8.53   38.71     0.99                 0 Within range
 TATASTEEL.NS    Basic Materials   190.20      21.49    